In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import torch.utils.checkpoint as checkpoint
import gc

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Clear memory before training
torch.cuda.empty_cache()
gc.collect()

# Data Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(30),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load Data
train_dir = "train"
valid_dir = "valid"

train_dataset = datasets.ImageFolder(train_dir, transform=transform)
valid_dataset = datasets.ImageFolder(valid_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

class_labels = train_dataset.classes
print(class_labels)

# **Modified ResNet-50 Model with Proper Checkpointing**
class CheckpointedResNet(nn.Module):
    def __init__(self, num_classes):
        super(CheckpointedResNet, self).__init__()
        self.model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Linear(num_ftrs, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        # **Apply checkpointing to memory-heavy layers**
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)

        # **Checkpointed layers**
        x = checkpoint.checkpoint(self.model.layer1, x)
        x = checkpoint.checkpoint(self.model.layer2, x)
        x = checkpoint.checkpoint(self.model.layer3, x)
        x = checkpoint.checkpoint(self.model.layer4, x)

        x = self.model.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.model.fc(x)
        return x

model = CheckpointedResNet(len(class_labels)).to(device)

# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# **Use Mixed Precision (AMP)**
scaler = torch.cuda.amp.GradScaler()

def free_memory():
    torch.cuda.empty_cache()
    gc.collect()

# Training Loop
epochs = 30
early_stopping_patience = 5
best_val_acc = 0
early_stop_counter = 0

for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad()

        # **Use mixed precision for memory efficiency**
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # **Delete unused variables to free memory**
        del images, labels, outputs, loss
        torch.cuda.empty_cache()

    train_acc = 100 * correct / total
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}, Accuracy: {train_acc:.2f}%")

    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model(images)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

            del images, labels, outputs  # Free memory after each batch
            torch.cuda.empty_cache()

    val_acc = 100 * val_correct / val_total
    print(f"Validation Accuracy: {val_acc:.2f}%")

    # Early Stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        early_stop_counter = 0
        torch.save(model.state_dict(), "best_model.pth")
        print("Best model saved!")
    else:
        early_stop_counter += 1

    if early_stop_counter >= early_stopping_patience:
        print("Early stopping triggered.")
        break

    free_memory()

# Load best model and evaluate
model.load_state_dict(torch.load("best_model.pth"))
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in valid_loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        del images, labels, outputs
        torch.cuda.empty_cache()

print(f"ResNet Model Accuracy: {100 * correct / total:.2f}%")
                                                                                                                        

Using device: cuda
['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 

/tmp/ipykernel_107785/3857899427.py:79: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_107785/3857899427.py:102: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1, Loss: 3.5401, Accuracy: 14.00%
Validation Accuracy: 21.89%
Best model saved!
Epoch 2, Loss: 3.4004, Accuracy: 28.29%
Validation Accuracy: 39.19%
Best model saved!
Epoch 3, Loss: 3.2744, Accuracy: 40.92%
Validation Accuracy: 48.42%
Best model saved!
Epoch 4, Loss: 3.1663, Accuracy: 51.73%
Validation Accuracy: 61.93%
Best model saved!
Epoch 5, Loss: 3.0774, Accuracy: 60.65%
Validation Accuracy: 67.95%
Best model saved!
Epoch 6, Loss: 3.0237, Accuracy: 65.97%
Validation Accuracy: 73.05%
Best model saved!
Epoch 7, Loss: 2.9858, Accuracy: 69.71%
Validation Accuracy: 75.10%
Best model saved!
Epoch 8, Loss: 2.9617, Accuracy: 72.11%
Validation Accuracy: 77.01%
Best model saved!
Epoch 9, Loss: 2.9383, Accuracy: 74.42%
Validation Accuracy: 79.53%
Best model saved!
Epoch 10, Loss: 2.9126, Accuracy: 77.03%
Validation Accuracy: 80.82%
Best model saved!
Epoch 11, Loss: 2.8960, Accuracy: 78.68%
Validation Accuracy: 83.46%
Best model saved!
Epoch 12, Loss: 2.8879, Accuracy: 79.43%
Validation 